# ??? ?? ?? ?? ?? ???

Auto MPG ???? ??? ??? ???? ?? ??(MPG)? ???? ?? ??? ?????.

?? ??? ???? ?? ??? ?? ?? 4?? ?????.

- ????
- ??????
- ?? ????
- KNN ??

? ??? R?, MAE, RMSE? ???? RMSE? ?? ?? ??? ?? ??? ?????.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeRegressor

## 1. ??? ????

Seaborn?? ???? `mpg.csv` ???? ?????. ????? `data/mpg.csv` ??? ??? ?? ??? ????, ??? ?? URL?? ??????.

In [ ]:
BASE_DIR = Path.cwd()
DATA_PATH = BASE_DIR / "data" / "mpg.csv"
DATA_URL = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/mpg.csv"

FEATURES = [
    "cylinders",
    "displacement",
    "horsepower",
    "weight",
    "acceleration",
    "model_year",
]

FEATURE_LABELS = {
    "cylinders": "??? ?",
    "displacement": "???",
    "horsepower": "??",
    "weight": "??",
    "acceleration": "???",
    "model_year": "??",
}

In [ ]:
if DATA_PATH.exists():
    df = pd.read_csv(DATA_PATH)
else:
    df = pd.read_csv(DATA_URL)
    DATA_PATH.parent.mkdir(exist_ok=True)
    df.to_csv(DATA_PATH, index=False)

df = df[["mpg", *FEATURES]].dropna()
df.head()

## 2. ??? ??

?? ??? `mpg`??, ?? ??? ??? ?, ???, ??, ??, ???, ?????.

In [ ]:
print(f"??? ??: {len(df)}?")
df.describe().round(2)

In [ ]:
df.isna().sum()

## 3. ?? ???? ??? ??? ??

?? ???? ?? ??? 80%, ??? ??? 20%? ????. `random_state=42`? ??? ??? ??? ?? ??? ???? ??????.

In [ ]:
X = df[FEATURES]
y = df["mpg"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"?? ???: {len(X_train)}?")
print(f"??? ???: {len(X_test)}?")

## 4. ?? ?? ??

???? ?? ?? ???? ????? ??? ?????. KNN ??? ?? ?? ????? `StandardScaler`? ???? ? ?????.

In [ ]:
models = {
    "????": LinearRegression(),
    "??????": DecisionTreeRegressor(random_state=42, max_depth=5),
    "?? ????": RandomForestRegressor(
        random_state=42,
        n_estimators=300,
        max_depth=8,
    ),
    "KNN ??": make_pipeline(
        StandardScaler(),
        KNeighborsRegressor(n_neighbors=5),
    ),
}

trained_models = {}
results = []

for model_name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))

    trained_models[model_name] = model
    results.append(
        {
            "??": model_name,
            "R2": round(r2_score(y_test, y_pred), 3),
            "MAE": round(mean_absolute_error(y_test, y_pred), 3),
            "RMSE": round(float(rmse), 3),
        }
    )

results_df = pd.DataFrame(results).sort_values("RMSE").reset_index(drop=True)
results_df

## 5. ?? ?? ??

RMSE? ?? ??? ????? ?? ???? ?? ?????. ??? RMSE? ?? ?? ??? ?? ??? ?????.

In [ ]:
best_model_name = results_df.loc[0, "??"]
best_model = trained_models[best_model_name]

print(f"?? ?? ??: {best_model_name}")
print(f"RMSE: {results_df.loc[0, 'RMSE']}")
print(f"R2: {results_df.loc[0, 'R2']}")
print(f"MAE: {results_df.loc[0, 'MAE']}")

## 6. ?? ?? ??

?? ??? ?? ?? ??? ???? ????? ?????.

- ?? ????? ??????? `feature_importances_`? ?????.
- ????? ?? ??? ???? ?????.
- KNN? ?? ??? ?? ???? ?? ???? ?? 0?? ?????.

In [ ]:
estimator = best_model
if hasattr(best_model, "named_steps"):
    estimator = list(best_model.named_steps.values())[-1]

if hasattr(estimator, "feature_importances_"):
    importance_values = estimator.feature_importances_
    importance_type = "feature_importances_"
elif hasattr(estimator, "coef_"):
    importance_values = np.abs(estimator.coef_)
    importance_type = "coefficient"
else:
    importance_values = np.zeros(len(FEATURES))
    importance_type = "not_supported"

importance_df = (
    pd.DataFrame(
        {
            "??": [FEATURE_LABELS[item] for item in FEATURES],
            "???": importance_values,
        }
    )
    .sort_values("???", ascending=False)
    .reset_index(drop=True)
)

print(f"??? ?? ??: {importance_type}")
importance_df

## 7. ?? ??? ??

Streamlit ??? ???? ?? ????? ?? ??? ??? ???.

In [ ]:
sample = pd.DataFrame(
    [[4, 140, 90, 2400, 15, 82]],
    columns=FEATURES,
)

prediction = round(float(best_model.predict(sample)[0]), 2)
print(f"?? ??: {prediction} MPG")

## 8. ??

? ?????? Auto MPG ???? ??? 4?? ?? ??? ???? ??? ??????. ?? Streamlit ?? ? ?? ??? ?? ???? ??? ??? ?, ???? ??? ??? ??? ?? ?? ??? ?????.